<a href="https://colab.research.google.com/github/Madihasafi/ITC-300-Data-Visualization-/blob/main/Madiha_Safi%2C_Template_of_w5p2_Exploratory_Data_Analysis_(EDA)_Part_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITC 300: Data Visualization
## Updated Laboratory Examples — Sections D, E, and F

**Dataset:** `contributors_with_candidate_id.txt`  
**Source lesson alignment:** Section 2 — Data Transformation  
**Environment:** Google Colab / Jupyter Notebook  
**Main libraries:** pandas, NumPy, Matplotlib

This notebook replaces the previous examples in **Section D**, **Section E**, and **Section F** with examples based on the campaign-contribution dataset. The examples follow the Section 2 slide flow:

`cleaning → Boolean masks and queries → sorting → distinct records → assignment → dropping → aggregation → groupby → split–apply–combine → index manipulation`


## Dataset variables used in the examples

The file is pipe-separated, therefore it must be loaded with `sep="|"`.

| Column | Meaning in this laboratory |
|---|---|
| `id` | Source row identifier |
| `last_name`, `first_name`, `middle_name` | Contributor name fields |
| `street_1`, `street_2`, `city`, `state`, `zip` | Contributor address fields |
| `amount` | Contribution amount; negative values can represent reversals/refunds |
| `date` | Contribution date |
| `candidate_id` | Candidate identifier |


# Load the contributors dataset


In [3]:
import pandas as pd
import numpy as np

# Import the contributors_with_candidate_id.txt, where the sep='|'
df = pd.read_csv("contributors_with_candidate_id.txt", sep="|")

#Converting the date attribute to date-time type
df['date'] = pd.to_datetime(df['date'], errors='coerce')

#Converting the candidate_id attribute to categorical type
df['candidate_id'] = df['candidate_id'].astype('category')

display(df.head())


,id,last_name,first_name,middle_name,street_1,street_2,city,state,zip,amount,date,candidate_id
0,1,Agee,Steven,NaN,549 Laurel Branch Road,NaN,Floyd,VA,24091,500.0,2007-06-30,16
1,2,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,250.0,2007-05-16,16
2,3,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,50.0,2007-06-18,16
3,4,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,100.0,2007-06-21,16
4,5,Akin,Charles,NaN,10187 Sugar Creek Road,NaN,Bentonville,AR,72712,100.0,2007-06-16,16


# Section D — Data Transformation

This section focuses on transforming a single DataFrame. The goal is to make the data easier to inspect, filter, summarize, and prepare for visualization.


## D1. Clean and inspect the dataset structure

Before applying transformations, check the DataFrame size, column names, data types, missing values, and a small sample of records.


In [4]:
# Number of rows and columns
print('Rows and columns:', df.shape)

print()

Rows and columns: (175, 12)



In [5]:
# The attribute labels
print('Column names:')
print(df.columns.tolist())

print()

Column names:
['id', 'last_name', 'first_name', 'middle_name', 'street_1', 'street_2', 'city', 'state', 'zip', 'amount', 'date', 'candidate_id']



In [6]:
# The attribute data types
print('Data types:')
print(df.dtypes)
print()

Data types:
id                       int64
last_name               object
first_name              object
middle_name             object
street_1                object
street_2                object
city                    object
state                   object
zip                      int64
amount                 float64
date            datetime64[ns]
candidate_id          category
dtype: object



In [7]:
# Missing values per attribute
print('Missing values per column:')
display(df.isna().sum().to_frame('missing_count'))


Missing values per column:


,missing_count
id,0
last_name,0
first_name,0
middle_name,109
street_1,0
street_2,167
city,0
state,1
zip,0
amount,0


## D2. Drop a technical identifier column from an analysis copy

The `id` column identifies the source row, but it is not needed when the analytical unit is the contribution record. Drop it from a copy so the original DataFrame remains available.


In [8]:
# Drop technical atted attributes
df_no_id = df.copy()
del df_no_id['id']

print('Original shape:', df.shape)
print('After deleting id:', df_no_id.shape)
display(df_no_id.head())


Original shape: (175, 12)
After deleting id: (175, 11)


,last_name,first_name,middle_name,street_1,street_2,city,state,zip,amount,date,candidate_id
0,Agee,Steven,NaN,549 Laurel Branch Road,NaN,Floyd,VA,24091,500.0,2007-06-30,16
1,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,250.0,2007-05-16,16
2,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,50.0,2007-06-18,16
3,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,100.0,2007-06-21,16
4,Akin,Charles,NaN,10187 Sugar Creek Road,NaN,Bentonville,AR,72712,100.0,2007-06-16,16


## D3. Create a Boolean mask for contributions below 400

A Boolean mask is a True/False Series aligned with DataFrame rows. `True` means the row satisfies the condition and can be kept by filtering.


In [9]:
df= df_no_id.copy()

#Boolean mask for contributions below 400 $
small_amount_mask = df['amount'] < 400

#Which contribution transactions have an amount below a specified threshold?
print('First 10 mask values:')
display(small_amount_mask.head(5).
        to_frame('amount_less_than_400'))


First 10 mask values:


,amount_less_than_400
0,False
1,True
2,True
3,True
4,True


In [10]:
#Which contribution transactions have an amount below a specified threshold?
# Contributors below 400 $

small_contributions = df[small_amount_mask]


print('Rows with amount < 400:', small_contributions.shape[0])
display(small_contributions[['last_name', 'first_name', 'state',
                             'amount', 'candidate_id']].head(5))

Rows with amount < 400: 132


,last_name,first_name,state,amount,candidate_id
1,Ahrens,Don,CA,250.0,16
2,Ahrens,Don,CA,50.0,16
3,Ahrens,Don,CA,100.0,16
4,Akin,Charles,AR,100.0,16
7,Aldridge,Brittni,DC,250.0,16


## D4. Summarize a Boolean mask and combine query conditions

Because `True` behaves like 1 and `False` behaves like 0, the mean of a Boolean mask gives the proportion of rows satisfying the condition.

The combined condition below keeps only Virginia (`VA`) records with `amount < 400`.


In [11]:
# What is the weight of small contributions?

proportion_small = small_amount_mask.mean()
print(f'Proportion of contributions below 400: {proportion_small:.2%}')

Proportion of contributions below 400: 75.43%


In [12]:
# What is the weight of small contributions on a specific state?

va_small = df[(df['state'] == 'VA') & (df['amount'] < 400)]
print('VA rows with amount < 400:', va_small.shape[0])
display(va_small[['last_name', 'first_name', 'city', 'state',
                  'amount', 'date', 'candidate_id']])

VA rows with amount < 400: 4


,last_name,first_name,city,state,amount,date,candidate_id
27,Buckheit,Bruce,FAIRFAX,VA,100.00,2007-09-19,20
77,Ranganath,Anoop,Charlottesville,VA,-100.00,2008-04-21,32
88,Perreault,Louise,Hampton,VA,-34.08,2008-04-21,32
145,ABDELLA,THOMAS,FAIRFAX,VA,50.00,2007-09-30,35


## D5. Use `query()` and filter missing or non-missing values

`query()` is a readable alternative for row filtering when column names are simple. Missing-value filters are useful during cleaning because missing `state` values can affect state-level aggregation.


In [13]:
# What are the small contributions on a specific state?
va_small_query = df.query("state == 'VA' and amount < 400")
display(va_small_query[['last_name', 'first_name', 'state', 'amount']])

,last_name,first_name,state,amount
27,Buckheit,Bruce,VA,100.00
77,Ranganath,Anoop,VA,-100.00
88,Perreault,Louise,VA,-34.08
145,ABDELLA,THOMAS,VA,50.00


In [14]:
# What are the transactions with missing values on a specific attribute?
missing_state = df[df['state'].isna()]
print('Rows with missing state:')
display(missing_state[['last_name', 'first_name', 'city', 'state',
                       'amount', 'candidate_id']])

Rows with missing state:


,last_name,first_name,city,state,amount,candidate_id
125,BOURNE,TRAVIS,BRASSCHATT,NaN,-500.0,35


In [15]:
# What are the transactions with non missing values on a specific attribute?
non_missing_state = df[df['state'].notna()]

print('Rows with non-missing state:')
display(non_missing_state[['last_name', 'first_name', 'state',
                           'amount']].head())


Rows with non-missing state:


,last_name,first_name,state,amount
0,Agee,Steven,VA,500.0
1,Ahrens,Don,CA,250.0
2,Ahrens,Don,CA,50.0
3,Ahrens,Don,CA,100.0
4,Akin,Charles,AR,100.0


In [16]:
# What are the transactions that have the amount between a specific range?
# Who are the contributors with contribution amount between a specific range?
range_mask = (df['amount'] >= 10) & (df['amount'] <= 50)
range_contributions = df[range_mask]

print('Small positive contributions between 10 and 50:')
display(range_contributions[['last_name', 'first_name', 'state',
                              'amount', 'candidate_id']])

Small positive contributions between 10 and 50:


,last_name,first_name,state,amount,candidate_id
2,Ahrens,Don,CA,50.0,16
18,Ardle,William,OH,50.0,16
25,Buckler,Steve,CA,50.0,20
26,Buckler,Steve,CA,25.0,20
34,Buck,Barbara,FL,50.0,20
35,Buck,Barbara,FL,50.0,20
38,Buchanek,Elizabeth,MD,50.0,20
49,Harrison,Ryan,CA,25.0,20
101,Aarons,Elaine,MA,25.0,34
104,Aaron,Shirley,FL,50.0,34


## D6. Sort records by contribution amount

Sorting helps reveal unusually small, negative, or high contribution amounts. In this dataset, negative values are important because they may represent reversals or refunds rather than ordinary positive contributions.


In [17]:
# What are the Smallest contribution amounts?
smallest_amounts = df.nsmallest(10, 'amount')

print('Smallest contribution amounts:')
display(smallest_amounts[['last_name', 'first_name', 'state',
                            'amount', 'date', 'candidate_id']])

Smallest contribution amounts:


,last_name,first_name,state,amount,date,candidate_id
90,Kazor,Christopher,IL,-2592.0,2008-04-21,32
29,Buckel,Linda,UT,-2300.0,2007-08-14,20
51,BYINGTON,MARGARET,MI,-2300.0,2008-03-03,22
52,BYERS,BOB,CA,-2300.0,2008-03-07,22
54,BUSH,KRYSTIE,CO,-2300.0,2008-03-06,22
55,BUSH,ERIC,CO,-2300.0,2008-03-06,22
56,BURTON,SUSAN,FL,-2300.0,2008-03-05,22
57,BURTON,STEVEN,FL,-2300.0,2008-03-05,22
58,BURTON,GLENN,FL,-2300.0,2008-03-05,22
64,BURKE,DONALD,CA,-2300.0,2008-03-11,22


In [18]:
# What is the meaning of negative amount?

negative_amounts = df[df['amount'] < 0]

print('Negative contribution records:')
display(negative_amounts[['last_name', 'first_name', 'state',
                          'amount', 'date', 'candidate_id']].head())

Negative contribution records:


,last_name,first_name,state,amount,date,candidate_id
29,Buckel,Linda,UT,-2300.0,2007-08-14,20
40,Buchanan,John,FL,-500.0,2007-09-24,20
43,Buchanan,John,FL,-700.0,2007-08-28,20
50,BYNUM,HERBERT,FL,-500.0,2008-03-10,22
51,BYINGTON,MARGARET,MI,-2300.0,2008-03-03,22


In [19]:
# Who are the highest donors?
largest_amounts = df.nlargest(10, 'amount')

print('Largest contribution amounts:')
display(largest_amounts[['last_name', 'first_name', 'state',
                           'amount', 'date', 'candidate_id']])


Largest contribution amounts:


,last_name,first_name,state,amount,date,candidate_id
30,Buckel,Linda,UT,4600.0,2007-08-14,20
159,ABATE,MARIA,FL,2600.0,2008-01-25,37
13,Altes,R.D.,AR,2300.0,2007-06-21,16
15,Anthony,John,AR,2300.0,2007-06-12,16
21,Baker,David,AR,2300.0,2007-04-11,16
28,Buckel,Linda,UT,2300.0,2007-08-14,20
33,Buck,Blaine,ME,2300.0,2007-09-30,20
135,ABRAMOWITZ,NIRA,CT,2300.0,2007-09-14,35
5,Akin,Mike,AR,1500.0,2007-05-18,16
9,Allen,John D.,SC,1300.0,2007-06-29,16


## D7. Select distinct contributors

The dataset can contain multiple contributions from the same person. If the question is about contributors rather than contributions, use `drop_duplicates()` on meaningful identity fields.


In [20]:
# What is the total number of values per name and last name attributes?

name_counts = df[['last_name', 'first_name']].count()

display(name_counts)

,0
last_name,175
first_name,175


In [21]:
# How many of these values are repeated?
# What are uniqie values (or distinc donors)?

name_counts = df[['last_name', 'first_name']].count()
distinct_name_counts = df[['last_name', 'first_name']].nunique()

comparison = pd.DataFrame({
    'all_rows_count': name_counts,
    'distinct_name_count': distinct_name_counts
})
display(comparison)

,all_rows_count,distinct_name_count
last_name,175,96
first_name,175,105


In [24]:
# What are uniqie values (or distinc donors)?


subset = df[["last_name", "first_name", "city", "state"]]
distinct_combinations = subset.drop_duplicates()
print("Distinct contributor/location combinations:\n", len(distinct_combinations))

# Display result
display(distinct_combinations.head())

Distinct contributor/location combinations:
 127


,last_name,first_name,city,state
0,Agee,Steven,Floyd,VA
1,Ahrens,Don,Pleasanton,CA
4,Akin,Charles,Bentonville,AR
5,Akin,Mike,Monticello,AR
6,Akin,Rebecca,Monticello,AR


## D8. Create and assign new columns

New columns can make the analytical table easier to read. Here we create a contributor label, contribution direction, and a simple amount band.


In [25]:
# Creating new columns

df['FullName'] = df['last_name'] + ',' + df['first_name']

df[['last_name','first_name','FullName']].head(5)


,last_name,first_name,FullName
0,Agee,Steven,"Agee,Steven"
1,Ahrens,Don,"Ahrens,Don"
2,Ahrens,Don,"Ahrens,Don"
3,Ahrens,Don,"Ahrens,Don"
4,Akin,Charles,"Akin,Charles"


In [26]:
#Adding a new column using assign function

df = df.assign(ucname = df['last_name'] + ':' + df['first_name'])

df[['last_name','first_name','ucname']].head(5)

,last_name,first_name,ucname
0,Agee,Steven,Agee:Steven
1,Ahrens,Don,Ahrens:Don
2,Ahrens,Don,Ahrens:Don
3,Ahrens,Don,Ahrens:Don
4,Akin,Charles,Akin:Charles


## D9. Conditional modification with `loc[]`

Use `loc[row_condition, column_name]` when modifying selected values. The example below standardizes missing `state` values as `UNKNOWN` in a copy.


In [27]:
# Find and replace with appropriate value the missing value record
df_modified = df.copy()

df_modified.loc[df_modified['state'].isna(), 'state'] = 'UNKNOWN'

print('Number of UNKNOWN state values after modification:',
 (df_modified['state'] == 'UNKNOWN').sum())

display(df_modified.loc[df_modified['state'] ==
'UNKNOWN', ['city', 'state', 'amount', 'candidate_id']])


Number of UNKNOWN state values after modification: 1


,city,state,amount,candidate_id
125,BRASSCHATT,UNKNOWN,-500.0,35


## D10. Drop columns and selected records

Drop columns that are not needed for the current analytical question. Drop records only when the reason is explicit. Below, negative contributions are removed from a specific analysis copy because the next question studies non-negative contribution behavior.


In [28]:
# Delete non essential attributes
main_attributes=df.copy()

display(main_attributes.head())

,last_name,first_name,middle_name,street_1,street_2,city,state,zip,amount,date,candidate_id,FullName,ucname
0,Agee,Steven,NaN,549 Laurel Branch Road,NaN,Floyd,VA,24091,500.0,2007-06-30,16,"Agee,Steven",Agee:Steven
1,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,250.0,2007-05-16,16,"Ahrens,Don",Ahrens:Don
2,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,50.0,2007-06-18,16,"Ahrens,Don",Ahrens:Don
3,Ahrens,Don,NaN,4034 Rennellwood Way,NaN,Pleasanton,CA,94566,100.0,2007-06-21,16,"Ahrens,Don",Ahrens:Don
4,Akin,Charles,NaN,10187 Sugar Creek Road,NaN,Bentonville,AR,72712,100.0,2007-06-16,16,"Akin,Charles",Akin:Charles


In [29]:
#Describe main attributes
display(main_attributes.describe())

,zip,amount,date
count,1.750000e+02,175.000000,175
mean,3.780014e+08,3.418114,2007-12-08 00:00:00
min,2.474000e+03,-2592.000000,2007-04-08 00:00:00
25%,9.336700e+04,-175.000000,2007-08-14 00:00:00
50%,3.233313e+08,100.000000,2008-01-21 00:00:00
75%,7.816946e+08,300.000000,2008-03-06 00:00:00
max,9.951532e+08,4600.000000,2008-11-20 00:00:00
std,3.628278e+08,1028.418999,NaN


In [30]:
# Find the value of maximum notation - Max Value
max_amount = main_attributes['amount'].max()
print(max_amount)

4600.0


In [31]:
# Who is the Highest Donor?
# Conditions using max
highest_donor = main_attributes[main_attributes['amount'] == main_attributes['amount'].max()][['first_name', 'last_name', 'amount', 'candidate_id']]
display(highest_donor)

,first_name,last_name,amount,candidate_id
30,Linda,Buckel,4600.0,20


In [32]:
# Analyze only some attributes
df_compact = main_attributes[['city', 'state', 'amount', 'date', 'candidate_id']].copy()
non_negative_df = df_compact[df_compact['amount'] >= 0].copy()


print('Compact columns:', df_compact.columns.tolist())
print('Rows before removing negative amounts:', df_compact.shape[0])
print('Rows after removing negative amounts:', non_negative_df.shape[0])
display(non_negative_df.head(10))


Compact columns: ['city', 'state', 'amount', 'date', 'candidate_id']
Rows before removing negative amounts: 175
Rows after removing negative amounts: 117


,city,state,amount,date,candidate_id
0,Floyd,VA,500.0,2007-06-30,16
1,Pleasanton,CA,250.0,2007-05-16,16
2,Pleasanton,CA,50.0,2007-06-18,16
3,Pleasanton,CA,100.0,2007-06-21,16
4,Bentonville,AR,100.0,2007-06-16,16
5,Monticello,AR,1500.0,2007-05-18,16
6,Monticello,AR,500.0,2007-05-18,16
7,Washington,DC,250.0,2007-06-06,16
8,North Augusta,SC,1000.0,2007-06-11,16
9,North Augusta,SC,1300.0,2007-06-29,16


# Section E — Grouping and Split–Apply–Combine

Section E uses `groupby()` to split the DataFrame into meaningful subsets, apply summary functions, and combine the results into a new analytical table.


## E1. Group by state

A `groupby` object stores the grouping rule. Summary calculations are applied later.


In [37]:
# What are the contributions by selected states?
state_group = df.groupby('state')

# Group contributions by state and describe the amount
state_describe = df[df['state'].isin(['AK', 'AR', 'AZ', 'CA', 'CO'])] \
    .groupby('state')['amount'].describe()

display(state_describe)



,count,mean,std,min,25%,50%,75%,max
state,,,,,,,,
AK,3.0,403.333333,100.166528,300.0,355.0,410.0,455.00,500.0
AR,12.0,1183.333333,775.574079,100.0,875.0,1000.0,1700.00,2300.0
AZ,1.0,120.000000,NaN,120.0,120.0,120.0,120.00,120.0
CA,23.0,-217.988261,942.102438,-2300.0,-175.0,50.0,225.00,1000.0
CO,4.0,-1455.750000,1025.166125,-2300.0,-2300.0,-1650.0,-805.75,-223.0


## E2. State-level descriptive statistics

Use `describe()` for a quick statistical overview by group. Use `sum()` when the question concerns total contribution amount.


In [38]:
# Analyze the contributions by state...

state_describe = df.groupby('state')['amount'].describe()

display(state_describe.head(10))

,count,mean,std,min,25%,50%,75%,max
state,,,,,,,,
AK,3.0,403.333333,100.166528,300.0,355.0,410.0,455.00,500.0
AR,12.0,1183.333333,775.574079,100.0,875.0,1000.0,1700.00,2300.0
AZ,1.0,120.000000,NaN,120.0,120.0,120.0,120.00,120.0
CA,23.0,-217.988261,942.102438,-2300.0,-175.0,50.0,225.00,1000.0
CO,4.0,-1455.750000,1025.166125,-2300.0,-2300.0,-1650.0,-805.75,-223.0
CT,1.0,2300.000000,NaN,2300.0,2300.0,2300.0,2300.00,2300.0
DC,5.0,-309.982000,529.644175,-1000.0,-500.0,-500.0,200.09,250.0
FL,30.0,-135.000000,1177.383862,-2300.0,-500.0,100.0,500.00,2600.0
IA,1.0,250.000000,NaN,250.0,250.0,250.0,250.00,250.0


In [41]:
# What are the total amounts of contributions per state?
# Which state has generated the largest amount?

# groupby - sum contributions by state
state_total = df.groupby('state')['amount'].sum().sort_values(ascending=False)
display(state_total)

,amount
state,
AR,14200.00
UT,5050.00
ME,2520.00
SC,2400.00
CT,2300.00
TX,1985.24
LA,1300.00
AK,1210.00
OK,800.00


## E3. Mean and standard deviation by state

Mean identifies typical contribution size by state, while standard deviation shows variability.


In [42]:
# groupby - mean contributions by state
state_mean = df.groupby('state')['amount'].mean().sort_values(ascending=False)
display(state_mean)

,amount
state,
CT,2300.000000
AR,1183.333333
SC,800.000000
LA,650.000000
ME,630.000000
NC,500.000000
UT,459.090909
AK,403.333333
OK,266.666667


In [43]:
# Create Boolean mask for non-negative amounts
non_negative_mask = df['amount'] >= 0

# Create a new DataFrame containing only non-negative amounts
non_negative_df = df[non_negative_mask].copy()

# groupby - mean contributions by state
state_mean_nn = non_negative_df.groupby('state')['amount'].mean()
display(state_mean_nn)


,amount
state,
AK,403.333333
AR,1183.333333
AZ,120.000000
CA,284.053333
CT,2300.000000
DC,225.045000
FL,483.333333
IA,250.000000
IL,125.000000


In [44]:
# What is the mean amount of contributions per state?
# What is the standart deviation of contributions per state?

state_mean = non_negative_df.groupby('state')['amount'].mean()
state_std = non_negative_df.groupby('state')['amount'].std()

state_mean_std = pd.concat(
    [state_mean.rename('mean_amount'), state_std.rename('std_amount')],
    axis=1
).sort_values('mean_amount', ascending=False)

display(state_mean_std.head(10))


,mean_amount,std_amount
state,,
CT,2300.000000,NaN
AR,1183.333333,775.574079
SC,800.000000,624.499800
UT,735.000000,1530.395737
LA,650.000000,212.132034
ME,630.000000,1113.522938
TX,508.333333,532.368920
NC,500.000000,NaN
FL,483.333333,608.344749


## E4. Group by candidate and state

Multiple grouping variables create a more detailed comparison. Here, each group is a candidate–state combination.


In [45]:
# What are the summary statistics of contributions for each cantidate?

candidate_state_summary = (
    non_negative_df
    .groupby(['candidate_id', 'state'], observed=True)
    .agg(
        contribution_count=('amount', 'count'),
        total_amount=('amount', 'sum'),
        mean_amount=('amount', 'mean')
    )
    .sort_values(['candidate_id', 'total_amount'], ascending=[True, False])
)

display(candidate_state_summary.head(15))


contribution_count  total_amount  mean_amount
candidate_id state                                               
16           AR                     12      14200.00  1183.333333
             SC                      2       2300.00  1150.000000
             CA                      4        650.00   162.500000
             OH                      3        650.00   216.666667
             NC                      1        500.00   500.000000
             VA                      1        500.00   500.000000
             DC                      1        250.00   250.000000
             IA                      1        250.00   250.000000
20           UT                      2       6900.00  3450.000000
             FL                      9       4800.00   533.333333
             ME                      1       2300.00  2300.000000
             CA                      5        660.80   132.160000
             DC                      1        200.09   200.090000
             IL                      1        200.00   200.000000
             MO                      1        100.00   100.000000

## E5. Split–apply–combine shown explicitly with a loop

This loop shows what `groupby()` does internally: split rows by state, apply calculations inside each subset, and combine the printed summaries.


In [46]:
# What are the summary statistics of contributions per state?

for state, group in non_negative_df.groupby('state'):
    print(
        'State:', state,
        '| records:', len(group),
        '| mean amount:', round(group['amount'].mean(), 2),
        '| total amount:', round(group['amount'].sum(), 2),
        '| standard deviation:', round(group['amount'].std(), 2)
    )

State: AK | records: 3 | mean amount: 403.33 | total amount: 1210.0 | standard deviation: 100.17
State: AR | records: 12 | mean amount: 1183.33 | total amount: 14200.0 | standard deviation: 775.57
State: AZ | records: 1 | mean amount: 120.0 | total amount: 120.0 | standard deviation: nan
State: CA | records: 15 | mean amount: 284.05 | total amount: 4260.8 | standard deviation: 324.67
State: CT | records: 1 | mean amount: 2300.0 | total amount: 2300.0 | standard deviation: nan
State: DC | records: 2 | mean amount: 225.04 | total amount: 450.09 | standard deviation: 35.29
State: FL | records: 21 | mean amount: 483.33 | total amount: 10150.0 | standard deviation: 608.34
State: IA | records: 1 | mean amount: 250.0 | total amount: 250.0 | standard deviation: nan
State: IL | records: 2 | mean amount: 125.0 | total amount: 250.0 | standard deviation: 106.07
State: LA | records: 2 | mean amount: 650.0 | total amount: 1300.0 | standard deviation: 212.13
State: MA | records: 5 | mean amount: 84.

## E6. Index manipulation with `set_index()` and `reset_index()`

`set_index()` relocates a column to become the row index. This can simplify label-based access. `reset_index()` moves the index back into a regular column.


In [47]:
df_indexed = non_negative_df.copy()
df_indexed = df_indexed.set_index('last_name')

print('Indexed by last_name:')
display(df_indexed[['first_name', 'state', 'amount','candidate_id']]
        .head())

Indexed by last_name:


,first_name,state,amount,candidate_id
last_name,,,,
Agee,Steven,VA,500.0,16
Ahrens,Don,CA,250.0,16
Ahrens,Don,CA,50.0,16
Ahrens,Don,CA,100.0,16
Akin,Charles,AR,100.0,16


In [48]:
print('Rows with last name Ahrens:')
ahrens_rows = df_indexed.loc[['Ahrens']]
display(ahrens_rows[['first_name', 'state', 'amount', 'date', 'candidate_id']])

Rows with last name Ahrens:


,first_name,state,amount,date,candidate_id
last_name,,,,,
Ahrens,Don,CA,250.0,2007-05-16,16
Ahrens,Don,CA,50.0,2007-06-18,16
Ahrens,Don,CA,100.0,2007-06-21,16


In [49]:

without_ahrens = df_indexed.drop('Ahrens')

print('Rows after dropping Ahrens from indexed copy:',
      without_ahrens.shape[0])

Rows after dropping Ahrens from indexed copy: 114


In [50]:
reset_version = without_ahrens.reset_index()

display(reset_version[['last_name', 'first_name', 'state', 'amount']]
        .head())

,last_name,first_name,state,amount
0,Agee,Steven,VA,500.0
1,Akin,Charles,AR,100.0
2,Akin,Mike,AR,1500.0
3,Akin,Rebecca,AR,500.0
4,Aldridge,Brittni,DC,250.0
